# Processmine — ML Analysis Walkthrough

**Anomaly detection · Concept drift · Next-activity prediction**

This notebook covers the Python ML layer of processmine. It runs independently
of the R notebook — it reads the same XES fixture directly, then builds a
larger synthetic log for the ML examples where 15 cases would be too few.

Prerequisites:
```
pip install -e '../python[ml]'
```

---
## 1 · Setup

In [ ]:
import sys
sys.path.insert(0, "../python")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from processmine_ml import (
    read_xes,
    read_eventlog_parquet,
    extract_case_features,
    detect_anomalies,
    extract_case_stream,
    detect_drift,
    NextActivityPredictor,
)

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

---
## 2 · Load the sample event log

Read the same order-to-cash fixture used by the R notebook.
If the R notebook has already written a Parquet file, reading that is faster.

In [ ]:
import pathlib

parquet_path = pathlib.Path("../data/sample_logs/bpi2012_sample.parquet")
xes_path     = pathlib.Path("../data/sample_logs/bpi2012_sample.xes")

if parquet_path.exists():
    log = read_eventlog_parquet(parquet_path)
    print("Loaded from Parquet")
else:
    log = read_xes(xes_path)
    print("Loaded from XES")

print(f"Cases  : {log['case_id'].nunique()}")
print(f"Events : {len(log)}")
print(f"Period : {log['timestamp'].min()} → {log['timestamp'].max()}")
log.head()

---
## 3 · Synthetic log for ML examples

15 cases is too few for anomaly detection and drift to behave meaningfully.
We generate a larger synthetic order-to-cash log:

- **Cases 0–79**: stable process, throughput ≈ 3–5 hours, 5 activities.
- **Cases 80–99**: 3 obvious outlier cases injected (very long, unusual activity counts).
- **Cases 100–179**: process degrades — throughput doubles to ≈ 8–12 hours.

In [ ]:
def make_synthetic_log(
    n_stable   : int   = 80,
    n_outliers : int   = 3,
    n_degraded : int   = 80,
    seed       : int   = 42,
) -> pd.DataFrame:
    rng  = np.random.default_rng(seed)
    base = pd.Timestamp("2024-01-01", tz="UTC")
    rows: list[dict] = []
    case_idx = 0

    def add_case(activities, hours_per_step, resources):
        nonlocal case_idx
        cid = f"C{case_idx:04d}"
        t   = base + pd.Timedelta(hours=case_idx * 6)
        for act, h, res in zip(activities, hours_per_step, resources):
            rows.append({"case_id": cid, "activity": act, "timestamp": t,
                         "resource": res, "lifecycle": "complete"})
            t += pd.Timedelta(hours=h)
        case_idx += 1

    happy_acts = ["Create Order", "Approve Order", "Pick Items", "Ship Items", "Close Order"]
    resources  = ["alice", "bob", "carol", "dave", "alice"]

    # Stable cases
    for _ in range(n_stable):
        steps = rng.uniform(0.5, 1.5, size=5).tolist()
        add_case(happy_acts, steps, resources)

    # Outlier cases: unusually long, many extra steps
    for i in range(n_outliers):
        acts  = happy_acts + [f"Rework_{j}" for j in range(8)]
        steps = [20.0] * len(acts)
        res   = resources + ["eve"] * 8
        add_case(acts, steps, res)

    # Degraded cases: throughput doubles
    for _ in range(n_degraded):
        steps = rng.uniform(1.5, 3.0, size=5).tolist()
        add_case(happy_acts, steps, resources)

    df = pd.DataFrame(rows)
    df["timestamp"] = df["timestamp"].dt.as_unit("us")
    return df


synth = make_synthetic_log()
print(f"Synthetic log: {synth['case_id'].nunique()} cases, {len(synth)} events")

---
## 4 · Case feature extraction

`extract_case_features` computes five numeric features per case:
throughput, event count, unique activities, unique resources, and resource entropy.
These are the inputs for both anomaly detection and any downstream modelling.

In [ ]:
features = extract_case_features(synth)
print(features.describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

for ax, col, colour in zip(
    axes,
    ["throughput_s", "n_events", "resource_entropy"],
    ["#4e79a7", "#76b7b2", "#f28e2b"],
):
    ax.hist(features[col], bins=25, color=colour, edgecolor="white", linewidth=0.4)
    ax.set_title(col.replace("_", " "))
    ax.set_xlabel("value")
    ax.set_ylabel("count")

fig.suptitle("Case feature distributions", y=1.02)
plt.tight_layout()
plt.show()

---
## 5 · Anomaly detection

`detect_anomalies` trains an Isolation Forest on the case features.
Cases with the lowest anomaly score (most isolated in feature space) are
flagged as anomalous.

We expect the three outlier cases to be flagged: they have very long throughput,
many unique activities, and high resource entropy.

In [ ]:
anomalies = detect_anomalies(synth, contamination=0.05, random_state=42)

flagged = anomalies[anomalies["is_anomaly"]]
print(f"Flagged: {len(flagged)} of {len(anomalies)} cases")
print()
print(flagged[["case_id", "throughput_s", "n_events", "n_unique_activities",
               "anomaly_score"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

normal  = anomalies[~anomalies["is_anomaly"]]
outlier = anomalies[ anomalies["is_anomaly"]]

ax.scatter(normal["throughput_s"],  normal["n_unique_activities"],
           c="#4e79a7", alpha=0.6, label="Normal",   s=30)
ax.scatter(outlier["throughput_s"], outlier["n_unique_activities"],
           c="#e15759", alpha=0.9, label="Anomaly",  s=80, marker="X")

ax.set_xlabel("Throughput (s)")
ax.set_ylabel("Unique activities")
ax.set_title("Isolation Forest — anomaly detection")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6 · Concept drift detection

`detect_drift` feeds a per-case metric stream into ADWIN (ADaptive WINdowing).
ADWIN signals a change point when two sub-windows of its adaptive window have
significantly different means.

We use the degraded log (without the outliers) so the step change in throughput
around case 80 is the dominant signal.

In [ ]:
# Build a log without the outlier cases for clean drift detection
outlier_ids = set(flagged["case_id"])
clean_log   = synth[~synth["case_id"].isin(outlier_ids)].copy()
print(f"Clean log: {clean_log['case_id'].nunique()} cases")

In [ ]:
drift_result = detect_drift(clean_log, metric="throughput", delta=0.002)

change_points = drift_result[drift_result["drift_detected"]]
print(f"Change points detected: {len(change_points)}")
if len(change_points):
    print(change_points[["case_id", "case_start", "throughput"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

x = range(len(drift_result))
ax.plot(x, drift_result["throughput"] / 3600,
        color="#4e79a7", linewidth=0.8, alpha=0.7, label="Throughput (h)")

for idx in change_points.index:
    ax.axvline(idx, color="#e15759", linewidth=1.5, linestyle="--", alpha=0.8)

if len(change_points):
    ax.axvline(change_points.index[0], color="#e15759", linewidth=1.5,
               linestyle="--", label="ADWIN change point")

ax.set_xlabel("Case (chronological)")
ax.set_ylabel("Throughput (hours)")
ax.set_title("ADWIN drift detection — throughput stream")
ax.legend()
plt.tight_layout()
plt.show()

### Activity-level drift

We can also monitor whether a specific activity's presence is changing over time.
Here we check whether `"Approve Order"` remains stable throughout the log.

In [ ]:
act_drift = detect_drift(clean_log, metric="activity:Approve Order", delta=0.002)
print("Activity drift change points:", act_drift["drift_detected"].sum())
# Approve Order appears in every case so ADWIN should detect no drift here
print("Binary stream mean:", act_drift["activity:Approve Order"].mean())

---
## 7 · Next-activity prediction

`NextActivityPredictor` trains a two-layer LSTM on all (prefix → next activity)
pairs extracted from the log. After fitting, `predict()` returns a ranked list
of likely next activities for any given prefix.

In [ ]:
predictor = NextActivityPredictor()
predictor.fit(
    clean_log,
    epochs       = 25,
    hidden_size  = 64,
    num_layers   = 2,
    embedding_dim= 32,
    lr           = 0.001,
    batch_size   = 64,
    random_state = 42,
)
print("Training complete.")
print("Vocabulary:", sorted(predictor._act_to_idx.keys()))

### Predict from a prefix

In [ ]:
prefix = ["Create Order"]
print(f"Prefix : {prefix}")
print(f"Top-3 next activities:")
print(predictor.predict(prefix, top_k=3).to_string(index=False))

In [ ]:
for prefix in [
    ["Create Order"],
    ["Create Order", "Approve Order"],
    ["Create Order", "Approve Order", "Pick Items"],
    ["Create Order", "Approve Order", "Pick Items", "Ship Items"],
]:
    top1 = predictor.predict(prefix, top_k=1)
    act  = top1["activity"].iloc[0]
    prob = top1["probability"].iloc[0]
    print(f"  {'→'.join(prefix):60s}  →  {act}  ({prob:.1%})")

### Evaluate accuracy

We evaluate on the training log itself to confirm the model has learned the
process structure. Held-out evaluation would require splitting by case.

In [ ]:
metrics = predictor.evaluate(clean_log)
print(f"Top-1 accuracy : {metrics['top1_accuracy']:.1%}")
print(f"Top-3 accuracy : {metrics['top3_accuracy']:.1%}")
print(f"Prefixes scored: {int(metrics['n_prefixes'])}")

Visualise the probability distribution over next activities at each step of the
happy-path sequence.

In [ ]:
happy_path = ["Create Order", "Approve Order", "Pick Items", "Ship Items", "Close Order"]
activities = sorted(predictor._act_to_idx.keys())

matrix = []
labels = []
for i in range(1, len(happy_path)):
    prefix = happy_path[:i]
    preds  = predictor.predict(prefix, top_k=len(activities))
    prob_map = dict(zip(preds["activity"], preds["probability"]))
    matrix.append([prob_map.get(a, 0.0) for a in activities])
    labels.append(" → ".join(prefix))

import numpy as np
mat = np.array(matrix)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(mat, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(activities)))
ax.set_xticklabels(activities, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_title("Predicted next-activity probabilities along the happy path")
plt.colorbar(im, ax=ax, label="Probability")
plt.tight_layout()
plt.show()

---
## 8 · Summary

| Module | Function | What it does |
|---|---|---|
| `io` | `read_xes`, `read_eventlog_parquet` | Load event logs from XES or Parquet |
| `anomaly` | `extract_case_features`, `detect_anomalies` | Isolation Forest on case-level features |
| `drift` | `extract_case_stream`, `detect_drift` | ADWIN change-point detection on metric streams |
| `prediction` | `NextActivityPredictor` | LSTM next-activity prediction with fit/predict/evaluate |

All functions accept validated processmine DataFrames. The schema contract
(`case_id`, `activity`, `timestamp[us, UTC]`) is enforced at every boundary.

See `ARCHITECTURE.md` for the full schema reference and cross-language contract.